Segmentation

In [ ]:
import os
import pandas as pd
import numpy as np
import cv2
import pydicom as pdcm
from tqdm import tqdm

def np_CountUpContinuingOnes(b_arr):
    # indice continuing zeros from left side.
    # ex: [0,1,1,0,1,0,0,1,1,1,0] -> [0,0,0,3,3,5,6,6,6,6,10]
    left = np.arange(len(b_arr))
    left[b_arr > 0] = 0
    left = np.maximum.accumulate(left)

    # from right side.
    # ex: [0,1,1,0,1,0,0,1,1,1,0] -> [0,3,3,3,5,5,6,10,10,10,10]
    rev_arr = b_arr[::-1]
    right = np.arange(len(rev_arr))
    right[rev_arr > 0] = 0
    right = np.maximum.accumulate(right)
    right = len(rev_arr) - 1 - right[::-1]

    return right - left - 1

def ExtractBreast(img, mask):
    img_copy = img.copy()
    mask_copy = mask.copy()
    

    img = np.where(img <= 20, 0, img)
    height, _ = img.shape


    y_a = height // 2 + int(height * 0.4)
    y_b = height // 2 - int(height * 0.4)
    b_arr = img[y_b:y_a].std(axis=0) != 0
    continuing_ones = np_CountUpContinuingOnes(b_arr)
    col_ind = np.where(continuing_ones == continuing_ones.max())[0]
    

    img = img[:, col_ind]
    mask = mask[:, col_ind]


    _, width = img.shape
    x_a = width // 2 + int(width * 0.4)
    x_b = width // 2 - int(width * 0.4)
    b_arr = img[:, x_b:x_a].std(axis=1) != 0
    continuing_ones = np_CountUpContinuingOnes(b_arr)
    row_ind = np.where(continuing_ones == continuing_ones.max())[0]
    

    img = img[row_ind, :]
    mask = mask[row_ind, :]

    return img_copy[row_ind][:, col_ind], mask_copy[row_ind][:, col_ind]

split_csv_path = "../segdetdata/segdet_split.csv"
split_df = pd.read_csv(split_csv_path)
split_df = split_df[split_df['dataset'] == 'CBIS-DDSM']


dataset_dir = "/Volumes/CBIS-DDSM_kaggle"#CBIS-DDSM data folder
df = pd.read_csv(f'{dataset_dir}/csv/dicom_info.csv')
df['image_path'] = df.image_path.apply(lambda x: x.replace('CBIS-DDSM', dataset_dir))
df = df[df['PatientID'].str.contains('mass', case=False)]


df = pd.merge(df, split_df[['data_name', 'data_split']], 
              left_on='PatientID', right_on='data_name', how='inner')


masks = {}


for index, row in tqdm(df.iterrows()):
    series_desc = row['SeriesDescription']
    patient_id = row['PatientID']
    
    if series_desc == 'full mammogram images':
        img_path = row['image_path']
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if patient_id in masks:
            if masks[patient_id].shape != img.shape:
                print(f"Skipping {patient_id} due to size mismatch.")
                del masks[patient_id]  
                continue
        else:
            mask = np.zeros_like(img, dtype=np.uint8)
            masks[patient_id] = mask
    elif series_desc == 'ROI mask images':
        base_patient_id = '_'.join(patient_id.split('_')[:-1])
        mask_path = row['image_path']
        roi_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if base_patient_id not in masks:
            masks[base_patient_id] = roi_mask
        else:
            if masks[base_patient_id].shape != roi_mask.shape:
                print(f"Skipping {base_patient_id} due to size mismatch.")
                del masks[base_patient_id] 
                continue
            masks[base_patient_id] = np.maximum(masks[base_patient_id], roi_mask)


target_dir = "../segdetdata/CBIS-DDSM"
for patient_id, mask in masks.items():

    data_split = df.loc[df['PatientID'] == patient_id, 'data_split'].values[0]
    
    img_path = df.loc[df['PatientID'] == patient_id, 'image_path'].values[0]
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    
    img, mask = ExtractBreast(img, mask)
    
    output_dir = os.path.join(target_dir, data_split, patient_id)
    os.makedirs(output_dir, exist_ok=True)
    
    img_output_path = os.path.join(output_dir, 'img.jpg')
    cv2.imwrite(img_output_path, cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8))
    
    mask_output_path = os.path.join(output_dir, 'mask.png')
    cv2.imwrite(mask_output_path, mask)
    
    print(f'Saved {patient_id} images and masks to {output_dir}')

Detection

In [ ]:
import os
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm

def find_cropped_image_position(full_image, cropped_image):
    res = cv2.matchTemplate(full_image, cropped_image, cv2.TM_CCOEFF_NORMED)
    min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(res)
    top_left = max_loc
    bottom_right = (top_left[0] + cropped_image.shape[1], top_left[1] + cropped_image.shape[0])
    return (*top_left, *bottom_right)


split_csv_path = "../segdetdata/segdet_split.csv"
split_df = pd.read_csv(split_csv_path)

split_df = split_df[split_df['dataset'] == 'CBIS-DDSM']

dataset_dir = "/Volumes/CBIS-DDSM_kaggle"#CBIS-DDSM data folder
df = pd.read_csv(f'{dataset_dir}/csv/dicom_info.csv')
df['image_path'] = df.image_path.apply(lambda x: x.replace('CBIS-DDSM', dataset_dir))
df = df[df['PatientID'].str.contains('mass', case=False)]


df = pd.merge(df, split_df[['data_name', 'data_split']], 
              left_on='PatientID', right_on='data_name', how='inner')


cropped = {}
data_path = '../segdetdata/CBIS-DDSM'
target_dir = "../segdetdata/CBIS-DDSM"


for index, row in tqdm(df.iterrows()):
    series_desc = row['SeriesDescription']
    patient_id = row['PatientID']
    data_split = row['data_split']
    
    if series_desc == 'cropped images':
        base_patient_id = '_'.join(patient_id.split('_')[:-1])
        cropped_path = row['image_path']
        cropped_image = cv2.imread(cropped_path, cv2.IMREAD_GRAYSCALE)
        

        original_image_path = os.path.join(data_path, data_split, base_patient_id, 'img.jpg')
        original_image = cv2.imread(original_image_path, cv2.IMREAD_GRAYSCALE)
        
        if original_image is None:
            continue
            
        boxes = find_cropped_image_position(original_image, cropped_image)
        
        if base_patient_id not in cropped:
            cropped[base_patient_id] = {
                'boxes': [boxes],
                'data_split': data_split
            }
        else:
            cropped[base_patient_id]['boxes'].append(boxes)


for patient_id, data in cropped.items():
    data_split = data['data_split']
    boxes = data['boxes']
    
    output_dir = os.path.join(target_dir, data_split, patient_id)
    os.makedirs(output_dir, exist_ok=True)
    
    bbox_output_path = os.path.join(output_dir, 'bboxes.npy')
    np.save(bbox_output_path, boxes)
    
    print(f'Saved {patient_id} bboxes to {output_dir}')

image-Classification

In [ ]:
import os
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
import pydicom as pdcm

def np_CountUpContinuingOnes(b_arr):
    # Calculate indices for continuing zeros from left side
    left = np.arange(len(b_arr))
    left[b_arr > 0] = 0
    left = np.maximum.accumulate(left)

    # Calculate indices from right side
    rev_arr = b_arr[::-1]
    right = np.arange(len(rev_arr))
    right[rev_arr > 0] = 0
    right = np.maximum.accumulate(right)
    right = len(rev_arr) - 1 - right[::-1]

    return right - left - 1

def ExtractBreast(img):
    img_copy = img.copy()
    img = np.where(img <= 40, 0, img)
    height, _ = img.shape

    y_a = height // 2 + int(height * 0.4)
    y_b = height // 2 - int(height * 0.4)
    b_arr = img[y_b:y_a].std(axis=0) != 0
    continuing_ones = np_CountUpContinuingOnes(b_arr)
    col_ind = np.where(continuing_ones == continuing_ones.max())[0]
    img = img[:, col_ind]

    _, width = img.shape
    x_a = width // 2 + int(width * 0.4)
    x_b = width // 2 - int(width * 0.4)
    b_arr = img[:, x_b:x_a].std(axis=1) != 0
    continuing_ones = np_CountUpContinuingOnes(b_arr)
    row_ind = np.where(continuing_ones == continuing_ones.max())[0]

    return img_copy[row_ind][:, col_ind]


split_csv_path = "../classification_data/classification_split.csv"
split_df = pd.read_csv(split_csv_path)

split_df = split_df[split_df['dataset'] == 'CBIS-DDSM-breast']


dataset_dir = "/Volumes/CBIS-DDSM_kaggle"#CBIS-DDSM data folder
df_dicom_info = pd.read_csv(f'{dataset_dir}/csv/dicom_info.csv')
df_dicom_info['image_path'] = df_dicom_info['image_path'].apply(lambda x: x.replace('CBIS-DDSM', dataset_dir))


description_files = [
    f"{dataset_dir}/csv/mass_case_description_train_set.csv",
    f"{dataset_dir}/csv/mass_case_description_test_set.csv",
    f"{dataset_dir}/csv/calc_case_description_train_set.csv",
    f"{dataset_dir}/csv/calc_case_description_test_set.csv"
]

additional_dfs = []
for file in description_files:
    df = pd.read_csv(file)

    df.rename(columns={'breast density': 'breast_density', 'breast_density': 'breast_density'}, inplace=True)
    additional_dfs.append(df)
additional_df = pd.concat(additional_dfs, ignore_index=True)


image_data = []


for index, row in tqdm(additional_df.iterrows(), total=len(additional_df)):
    patient_id = row['image file path'].split('/')[0]
    img_row = df_dicom_info[df_dicom_info['image_path'].apply(lambda x: x.split('/')[-2]) == row['image file path'].split('/')[-2]]
    
    if img_row.empty:
        print(f"No image info for patient_id: {patient_id}")
        continue

    img_path = img_row['image_path'].values[0]
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    
    if img is None:
        print(f"Failed to read image: {img_path}")
        continue
    
    img = ExtractBreast(img)

    info_dict = {
        "patient_id": patient_id,
        "img_path": img_path,
        "breast_density": str(row['breast_density']).replace(' ', ''),
        "left_or_right_breast": str(row['left or right breast']).replace(' ', ''),
        "image_view": str(row['image view']).replace(' ', ''),
        "abnormality_type": str(row['abnormality type']).replace(' ', ''),
        "assessment": str(row['assessment']).replace(' ', ''),
        "pathology": str(row['pathology']).replace(' ', ''),
    }
    
    image_data.append(info_dict)


image_df = pd.DataFrame(image_data)


image_df = pd.merge(image_df, split_df[['data_name', 'data_split']], 
                   left_on='patient_id', right_on='data_name', how='inner')

def save_images_and_info(df, output_base_dir):
    for _, row in df.iterrows():
        patient_id = row['patient_id']
        data_split = row['data_split']
        

        output_dir = os.path.join(output_base_dir, data_split, patient_id)
        os.makedirs(output_dir, exist_ok=True)
        

        img = cv2.imread(row['img_path'], cv2.IMREAD_GRAYSCALE)
        img = ExtractBreast(img)
        img_output_path = os.path.join(output_dir, 'img.jpg')
        cv2.imwrite(img_output_path, cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8))
        

        info_dict = {
            "Composition": row["breast_density"],
            "Bi-Rads": row["assessment"],
        }
        

        if 'Composition' in info_dict:
            acr_map = {'1': 'Level A', '2': 'Level B', '3': 'Level C', '4': 'Level D'}
            if info_dict['Composition'] in acr_map:
                info_dict['Composition'] = acr_map[info_dict['Composition']]
        

        if 'Bi-Rads' in info_dict:
            if info_dict['Bi-Rads'] in ['0','1', '2', '3', '5', '4']:
                info_dict['Bi-Rads'] = f"Bi-Rads {info_dict['Bi-Rads']}"
        
        dict_output_path = os.path.join(output_dir, 'info_dict.npy')
        np.save(dict_output_path, info_dict)
        
        print(f"Saved {patient_id} images and categories to {output_dir}")


output_base_dir = "../classification_data/CBIS-DDSM-breast"
save_images_and_info(image_df, output_base_dir)

print("Processing complete.")

cropped-classification

In [ ]:
import os
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
import pydicom as pdcm


split_csv_path = "../classification_data/classification_split.csv"
split_df = pd.read_csv(split_csv_path)

split_df = split_df[split_df['dataset'] == 'CBIS-DDSM-finding']


dataset_dir = "/Volumes/CBIS-DDSM_kaggle"
df = pd.read_csv(f'{dataset_dir}/csv/dicom_info.csv')
df['image_path'] = df['image_path'].apply(lambda x: x.replace('CBIS-DDSM', dataset_dir))


description_files = [
    f"{dataset_dir}/csv/mass_case_description_train_set.csv",
    f"{dataset_dir}/csv/mass_case_description_test_set.csv",
    f"{dataset_dir}/csv/calc_case_description_train_set.csv",
    f"{dataset_dir}/csv/calc_case_description_test_set.csv"
]
additional_dfs = [pd.read_csv(file) for file in description_files]
additional_df = pd.concat(additional_dfs, ignore_index=True)


all_data = []
for index, row in tqdm(additional_df.iterrows(), total=len(additional_df)):
    patient_id = row['cropped image file path'].split('/')[0]
    img_row = df[df['image_path'].apply(lambda x: x.split('/')[-2]) == row['cropped image file path'].split('/')[-2]]
    
    if img_row.empty:
        print(f"No image info for patient_id: {patient_id}")
        continue

    try:
        img_path = img_row[img_row['SeriesDescription'] == 'cropped images']['image_path'].values[0]
    except:
        print(f"Multiple or no entries found for patient_id: {patient_id}")
        continue


    all_data.append({
        "patient_id": patient_id,
        "img_path": img_path,
        "abnormality_type": str(row['abnormality type']).replace(' ', ''),
        "pathology": str(row['pathology']).replace(' ', '')
    })


all_data_df = pd.DataFrame(all_data)


all_data_df = pd.merge(all_data_df, split_df[['data_name', 'data_split']], 
                      left_on='patient_id', right_on='data_name', how='inner')

def save_images_and_info(df, output_base_dir):
    for _, row in df.iterrows():
        patient_id = row['patient_id']
        data_split = row['data_split']
        

        output_dir = os.path.join(output_base_dir, data_split, patient_id)
        os.makedirs(output_dir, exist_ok=True)


        img = cv2.imread(row['img_path'], cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"Failed to read image: {row['img_path']}")
            continue
        
        img_output_path = os.path.join(output_dir, 'img.jpg')
        cv2.imwrite(img_output_path, cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8))
        

        info_dict = {
            "Finding": [row["abnormality_type"]],
            "Pathology": row["pathology"]
        }
        info_dict = {k: v for k, v in info_dict.items() if v is not None}  # 删除为空的键
        if 'Pathology' in info_dict:
            if info_dict['Pathology'] == 'MALIGNANT':
                info_dict['Pathology'] = 'Malignant'
            elif info_dict['Pathology'] == 'BENIGN':
                info_dict['Pathology'] = 'Benign'
            elif info_dict['Pathology'] == "BENIGN_WITHOUT_CALLBACK":
                info_dict['Pathology'] = "Benign"

        dict_output_path = os.path.join(output_dir, 'info_dict.npy')
        np.save(dict_output_path, info_dict)
        
        print(f'Saved {patient_id} image and categories to {output_dir}')

output_base_dir = "../classification_data/CBIS-DDSM-finding"
save_images_and_info(all_data_df, output_base_dir)

print("Processing complete.")